# 03. Merge and Quality Control

This notebook loads the raw NFHS-5 data using memory-efficient chunked loading, filters for Tamil Nadu (state code 33), merges the Kids, Individual, and Household recodes, and conducts initial quality control.

In [1]:
import os
import pandas as pd
import pyreadstat

ModuleNotFoundError: No module named 'pyreadstat'

## 1. Load Data with Chunked Reading & Tamil Nadu Filtering

To handle the large raw datasets (~5.2 GB) without running out of memory (OOM), we load each file in chunks, filter for Tamil Nadu (state code 33) immediately, and select only key columns.

In [ ]:
raw_dir = os.path.join("..", "data", "raw")
kr_path = os.path.join(raw_dir, "IAKR7EFL.DTA")
ir_path = os.path.join(raw_dir, "IAIR7EFL.DTA")
hr_path = os.path.join(raw_dir, "IAHR7EFL.DTA")

# Check if files exist
for path in [kr_path, ir_path, hr_path]:
    assert os.path.exists(path), f"Raw file not found at {path}"

def read_dta_chunked(filepath, usecols, filter_col, filter_val, chunksize=50000):
    chunks = []
    total_raw_rows = 0
    for df, _ in pyreadstat.read_file_in_chunks(pyreadstat.read_dta, filepath, chunksize=chunksize, usecols=usecols):
        total_raw_rows += len(df)
        filtered = df[df[filter_col] == filter_val].copy()
        chunks.append(filtered)
    merged_df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=usecols)
    return merged_df, total_raw_rows

tn_code = 33

print("Loading Kids Recode (KR)...")
df_kr_tn, kr_raw = read_dta_chunked(kr_path, ["caseid", "midx", "v001", "v002", "v024", "hw70"], "v024", tn_code)

print("Loading Individual Recode (IR)...")
df_ir_tn, ir_raw = read_dta_chunked(ir_path, ["caseid", "v024", "v106", "v190", "v445"], "v024", tn_code)

print("Loading Household Recode (HR)...")
df_hr_tn, hr_raw = read_dta_chunked(hr_path, ["hhid", "hv001", "hv002", "hv024", "hv205", "hv206", "hv225"], "hv024", tn_code)

print(f"Loaded raw dataset row counts - KR: {kr_raw}, IR: {ir_raw}, HR: {hr_raw}")
print(f"After Tamil Nadu filtering row counts - KR: {df_kr_tn.shape[0]}, IR: {df_ir_tn.shape[0]}, HR: {df_hr_tn.shape[0]}")

## 2. Merge Datasets

We merge the child record with the mother's record using `caseid`, then join with the household record using cluster number (`v001`/`hv001`) and household number (`v002`/`hv002`).

In [ ]:
# Drop redundant state column from IR before merging
df_ir_merge = df_ir_tn.drop(columns=["v024"])
merged_child_mother = pd.merge(df_kr_tn, df_ir_merge, on="caseid", how="inner")
print(f"Merged KR and IR row count: {merged_child_mother.shape[0]}")

# Rename household ID variables to align with kids
df_hr_tn = df_hr_tn.rename(columns={"hv001": "v001", "hv002": "v002"})
df_hr_tn = df_hr_tn.drop(columns=["hv024"])

df_final = pd.merge(merged_child_mother, df_hr_tn, on=["v001", "v002"], how="inner")
print(f"Final merged shape: {df_final.shape}")

## 3. Rename Columns and Export

In [ ]:
rename_mapping = {
    "v024": "state",
    "v106": "mother_education",
    "v190": "wealth_index",
    "v445": "mother_bmi",
    "hv206": "electricity",
    "hv205": "toilet_type",
    "hv225": "share_toilet",
    "hw70": "stunting_haz"
}

df_final = df_final.rename(columns=rename_mapping)

output_dir = os.path.join("..", "data", "processed")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "tamilnadu_nfhs5_merged.csv")
df_final.to_csv(output_path, index=False)
print(f"Saved final merged dataset to: {output_path}")

## 4. Quality Control and Missing Values

In [ ]:
print("Missing values summary:")
print(df_final.isnull().sum())

print("\nStunting HAZ summary statistics:")
print(df_final['stunting_haz'].describe())